In [ ]:
import os, sys, time
sys.path.append('..')
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-5"

## Parte 1: Temperatura

In [ ]:
def ask_with_temperature(prompt: str, temperature: float, n: int = 3) -> list[str]:
    """Hace `n` peticiones con la misma temperatura para comparar variabilidad."""
    responses = []
    for _ in range(n):
        resp = client.messages.create(
            model=MODEL,
            max_tokens=150,
            temperature=temperature,
            messages=[{"role": "user", "content": prompt}]
        )
        responses.append(resp.content[0].text.strip())
    return responses

prompt = "Escribe un título creativo para una novela de ciencia ficción."

print("=== Temperature = 0.0 (Determinista) ===")
for r in ask_with_temperature(prompt, 0.0):
    print(f"  - {r}")

print("\n=== Temperature = 1.0 (Creativo) ===")
for r in ask_with_temperature(prompt, 1.0):
    print(f"  - {r}")

In [ ]:
# Temperatura para tareas de código (baja temperatura recomendada)
print("=== Temperatura 0.0 para código ===")
resp = client.messages.create(
    model=MODEL,
    max_tokens=300,
    temperature=0.0,
    messages=[{"role": "user", "content": 
               "Escribe una función Python que calcule el factorial de un número."}]
)
print(resp.content[0].text)

## Parte 2: Response Streaming

In [ ]:
import sys

print("=== Streaming con context manager ===")
print("Claude: ", end="", flush=True)

with client.messages.stream(
    model=MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": "Explica qué es la inteligencia artificial en 3 párrafos."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print()  # Nueva línea al final

In [ ]:
# Streaming con eventos detallados
print("=== Streaming con eventos ===")
full_text = ""
token_count = 0

with client.messages.stream(
    model=MODEL,
    max_tokens=200,
    messages=[{"role": "user", "content": "Escribe un haiku sobre la programacion."}]
) as stream:
    for event in stream:
        if hasattr(event, 'type'):
            if event.type == 'content_block_delta':
                if hasattr(event.delta, 'text'):
                    print(event.delta.text, end="", flush=True)
                    full_text += event.delta.text
                    token_count += 1

print(f"\n\nEstadisticas: ~{token_count} fragmentos recibidos")
print(f"Texto completo guardado: {len(full_text)} caracteres")


In [ ]:
# Comparar tiempo: streaming vs no-streaming
prompt = "Escribe un cuento corto de 100 palabras sobre un robot que aprende a cocinar."

# Sin streaming: espera hasta tener toda la respuesta
t0 = time.time()
resp = client.messages.create(
    model=MODEL, max_tokens=300,
    messages=[{"role": "user", "content": prompt}]
)
t_no_stream = time.time() - t0
print(f"Sin streaming: {t_no_stream:.2f}s para recibir {resp.usage.output_tokens} tokens")

# Con streaming: primer token llega antes
t0 = time.time()
first_token_time = None
print("\nCon streaming: ", end="", flush=True)
with client.messages.stream(
    model=MODEL, max_tokens=300,
    messages=[{"role": "user", "content": prompt}]
) as stream:
    for text in stream.text_stream:
        if first_token_time is None:
            first_token_time = time.time() - t0
        print(text, end="", flush=True)

print(f"\n\nTiempo al primer token: {first_token_time:.2f}s")
